# PyPowSyBl Network, Own Python PF, Security, And ML Tables

This notebook treats PyPowSyBl as the network-data source and validation simulator. The own Python model converts a PyPowSyBl `Network` into an internal per-unit `PowerCase`, then solves DC PF and Newton AC PF.

**Unit convention warning:** PyPowSyBl exposes many values in engineering units. The adapter converts MW/Mvar/kV/ohm/siemens into the internal per-unit convention used by the Julia model.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import pypowsybl.network as pn
import pypowsybl.loadflow as lf

notebook_dir = Path.cwd()
python_dir = notebook_dir.parent if notebook_dir.name == "notebooks" else notebook_dir / "python"
repo_root = python_dir.parent

sys.path.insert(0, str(python_dir))
sys.path.insert(0, str(python_dir / "tools" / "pypowsybl" / "src"))

from own_models.src.adapters.pypowsybl_adapter import pypowsybl_to_powercase
from own_models.src.analysis.compare_with_pypowsybl import compare_with_pypowsybl
from own_models.src.analysis.metrics import case_summary, solution_metrics
from own_models.src.pf.dc_power_flow import solve_dc_power_flow
from own_models.src.pf.ac_power_flow import solve_ac_power_flow_newton
from network_utils import network_tables
from security_utils import run_n_minus_one_branch_study

## M0: Inspect PyPowSyBl IEEE 14 Tables

Recreate the network before each experiment because PyPowSyBl load flow mutates the network object.

In [2]:
network_raw = pn.create_ieee14()
tables = network_tables(network_raw)

for name, table in tables.items():
    print(name, table.shape)

display(tables["buses"].head())
display(tables["lines"].head())
display(tables["two_windings_transformers"].head())
display(tables["generators"].head())
display(tables["loads"].head())
display(tables["shunts"].head())
display(tables["voltage_levels"].head())

buses (14, 7)
lines (17, 20)
two_windings_transformers (3, 21)
generators (5, 20)
loads (11, 11)
shunts (1, 18)
voltage_levels (14, 6)
substations (11, 5)


,id,name,v_mag,v_angle,connected_component,synchronous_component,voltage_level_id
0,VL1_0,,143.100,0.00,0,0,VL1
1,VL2_0,,141.075,-4.98,0,0,VL2
2,VL3_0,,136.350,-12.72,0,0,VL3
3,VL4_0,,137.565,-10.33,0,0,VL4
4,VL5_0,,137.700,-8.78,0,0,VL5


,id,name,r,x,g1,b1,g2,b2,p1,q1,i1,p2,q2,i2,voltage_level1_id,voltage_level2_id,bus1_id,bus2_id,connected1,connected2
0,L1-2-1,,3.532005,10.783732,0.0,0.000145,0.0,0.000145,NaN,NaN,NaN,NaN,NaN,NaN,VL1,VL2,VL1_0,VL2_0,True,True
1,L1-5-1,,9.846967,40.649040,0.0,0.000135,0.0,0.000135,NaN,NaN,NaN,NaN,NaN,NaN,VL1,VL5,VL1_0,VL5_0,True,True
2,L2-3-1,,8.563927,36.080033,0.0,0.000120,0.0,0.000120,NaN,NaN,NaN,NaN,NaN,NaN,VL2,VL3,VL2_0,VL3_0,True,True
3,L2-4-1,,10.590547,32.134320,0.0,0.000093,0.0,0.000093,NaN,NaN,NaN,NaN,NaN,NaN,VL2,VL4,VL2_0,VL4_0,True,True
4,L2-5-1,,10.379137,31.689630,0.0,0.000095,0.0,0.000095,NaN,NaN,NaN,NaN,NaN,NaN,VL2,VL5,VL2_0,VL5_0,True,True


,id,name,r,x,g,b,rated_u1,rated_u2,rated_s,p1,...,i1,p2,q2,i2,voltage_level1_id,voltage_level2_id,bus1_id,bus2_id,connected1,connected2
0,T4-7-1,,0.0,0.409875,0.0,0.0,132.030,14.0,NaN,NaN,...,NaN,NaN,NaN,NaN,VL4,VL7,VL4_0,VL7_0,True,True
1,T4-9-1,,0.0,0.800899,0.0,0.0,130.815,12.0,NaN,NaN,...,NaN,NaN,NaN,NaN,VL4,VL9,VL4_0,VL9_0,True,True
2,T5-6-1,,0.0,0.362909,0.0,0.0,125.820,12.0,NaN,NaN,...,NaN,NaN,NaN,NaN,VL5,VL6,VL5_0,VL6_0,True,True


,id,name,energy_source,target_p,min_p,max_p,min_q,max_q,rated_s,reactive_limits_kind,target_v,target_q,voltage_regulator_on,regulated_element_id,p,q,i,voltage_level_id,bus_id,connected
0,B1-G,,OTHER,232.4,-9999.0,9999.0,-1.797693e+308,1.797693e+308,NaN,MIN_MAX,143.100,-16.9,True,B1-G,NaN,NaN,NaN,VL1,VL1_0,True
1,B2-G,,OTHER,40.0,-9999.0,9999.0,-4.000000e+01,5.000000e+01,NaN,MIN_MAX,141.075,42.4,True,B2-G,NaN,NaN,NaN,VL2,VL2_0,True
2,B3-G,,OTHER,0.0,-9999.0,9999.0,0.000000e+00,4.000000e+01,NaN,MIN_MAX,136.350,23.4,True,B3-G,NaN,NaN,NaN,VL3,VL3_0,True
3,B6-G,,OTHER,0.0,-9999.0,9999.0,-6.000000e+00,2.400000e+01,NaN,MIN_MAX,12.840,12.2,True,B6-G,NaN,NaN,NaN,VL6,VL6_0,True
4,B8-G,,OTHER,0.0,-9999.0,9999.0,-6.000000e+00,2.400000e+01,NaN,MIN_MAX,21.800,17.4,True,B8-G,NaN,NaN,NaN,VL8,VL8_0,True


,id,name,type,p0,q0,p,q,i,voltage_level_id,bus_id,connected
0,B2-L,,UNDEFINED,21.7,12.7,NaN,NaN,NaN,VL2,VL2_0,True
1,B3-L,,UNDEFINED,94.2,19.0,NaN,NaN,NaN,VL3,VL3_0,True
2,B4-L,,UNDEFINED,47.8,-3.9,NaN,NaN,NaN,VL4,VL4_0,True
3,B5-L,,UNDEFINED,7.6,1.6,NaN,NaN,NaN,VL5,VL5_0,True
4,B6-L,,UNDEFINED,11.2,7.5,NaN,NaN,NaN,VL6,VL6_0,True


,id,name,g,b,model_type,max_section_count,section_count,solved_section_count,voltage_regulation_on,target_v,target_deadband,regulating_bus_id,p,q,i,voltage_level_id,bus_id,connected
0,B9-SH,,0.0,0.131944,LINEAR,1,1,NaN,False,NaN,NaN,VL9_0,NaN,NaN,NaN,VL9,VL9_0,True


,id,name,substation_id,nominal_v,high_voltage_limit,low_voltage_limit
0,VL1,,S1,135.0,NaN,NaN
1,VL2,,S2,135.0,NaN,NaN
2,VL3,,S3,135.0,NaN,NaN
3,VL4,,S4,135.0,NaN,NaN
4,VL5,,S5,135.0,NaN,NaN


## M1-M3: Convert PyPowSyBl Network To Internal PowerCase

The adapter extracts buses, lines, two-winding transformers, generators, loads, and shunts. Connected equipment that cannot be represented should raise an explicit error rather than being silently ignored.

In [3]:
print("nominal_apparent_power:", getattr(network_raw, "nominal_apparent_power", None))

print(network_raw.get_generators()[["target_p", "min_p", "max_p", "target_q", "min_q", "max_q"]].head())
print(network_raw.get_loads()[["p0", "q0"]].head())
print(network_raw.get_lines()[["r", "x", "g1", "b1", "g2", "b2"]].head())
print(network_raw.get_voltage_levels()[["nominal_v"]].head())

nominal_apparent_power: 100.0
      target_p   min_p   max_p  target_q          min_q          max_q
id                                                                    
B1-G     232.4 -9999.0  9999.0     -16.9 -1.797693e+308  1.797693e+308
B2-G      40.0 -9999.0  9999.0      42.4  -4.000000e+01   5.000000e+01
B3-G       0.0 -9999.0  9999.0      23.4   0.000000e+00   4.000000e+01
B6-G       0.0 -9999.0  9999.0      12.2  -6.000000e+00   2.400000e+01
B8-G       0.0 -9999.0  9999.0      17.4  -6.000000e+00   2.400000e+01
        p0    q0
id              
B2-L  21.7  12.7
B3-L  94.2  19.0
B4-L  47.8  -3.9
B5-L   7.6   1.6
B6-L  11.2   7.5
                r          x   g1        b1   g2        b2
id                                                        
L1-2-1   3.532005  10.783732  0.0  0.000145  0.0  0.000145
L1-5-1   9.846967  40.649040  0.0  0.000135  0.0  0.000135
L2-3-1   8.563927  36.080033  0.0  0.000120  0.0  0.000120
L2-4-1  10.590547  32.134320  0.0  0.000093  0.0  0.000093


In [4]:
case = pypowsybl_to_powercase(network_raw, base_mva=100.0)

print("base_mva:", case.base_mva)
for g in case.generators[:10]:
    print(g.id, "pg0_pu=", g.pg0, "pmax_pu=", g.pmax, "cost=", g.cost_c2, g.cost_c1, g.cost_c0)

for br in case.branches[:10]:
    print(br.id, "r_pu=", br.r, "x_pu=", br.x, "rate_pu=", br.rate_a, "tap=", br.tap)

base_mva: 100.0
B1-G pg0_pu= 2.324 pmax_pu= 2.5899999999999994 cost= 0.01 11.0 0.0
B2-G pg0_pu= 0.4 pmax_pu= 2.5899999999999994 cost= 0.02 12.0 0.0
B3-G pg0_pu= 0.0 pmax_pu= 2.5899999999999994 cost= 0.03 13.0 0.0
B6-G pg0_pu= 0.0 pmax_pu= 2.5899999999999994 cost= 0.04 14.0 0.0
B8-G pg0_pu= 0.0 pmax_pu= 2.5899999999999994 cost= 0.05 15.0 0.0
L1-2-1 r_pu= 0.01938 x_pu= 0.05916999999999999 rate_pu= 0.0 tap= 1.0
L1-5-1 r_pu= 0.05403 x_pu= 0.22304 rate_pu= 0.0 tap= 1.0
L10-11-1 r_pu= 0.08205 x_pu= 0.19207 rate_pu= 0.0 tap= 1.0
L12-13-1 r_pu= 0.22092 x_pu= 0.19987999999999997 rate_pu= 0.0 tap= 1.0
L13-14-1 r_pu= 0.17093 x_pu= 0.34802 rate_pu= 0.0 tap= 1.0
L2-3-1 r_pu= 0.04698999999999999 x_pu= 0.19797 rate_pu= 0.0 tap= 1.0
L2-4-1 r_pu= 0.058109999999999995 x_pu= 0.17632 rate_pu= 0.0 tap= 1.0
L2-5-1 r_pu= 0.056949999999999994 x_pu= 0.17388 rate_pu= 0.0 tap= 1.0
L3-4-1 r_pu= 0.06701 x_pu= 0.17102999999999996 rate_pu= 0.0 tap= 1.0
L4-5-1 r_pu= 0.01335 x_pu= 0.04211 rate_pu= 0.0 tap= 1.0


In [7]:
network_for_case = pn.create_ieee14()
case = pypowsybl_to_powercase(network_for_case, name="ieee14", base_mva=100.0)

case_summary_df = case_summary(case)
def audit_pypowsybl_units(network, case):
    print("network nominal_apparent_power:", getattr(network, "nominal_apparent_power", None))
    print("PowerCase base_mva:", case.base_mva)

    print("\nRaw PyPowSyBl generators:")
    print(network.get_generators().head())

    print("\nRaw PyPowSyBl loads:")
    print(network.get_loads().head())

    print("\nRaw PyPowSyBl lines:")
    print(network.get_lines().head())

    print("\nConverted PowerCase generators:")
    for g in case.generators[:10]:
        print(g.id, g.pg0, g.pmin, g.pmax, g.qg0, g.qmin, g.qmax, g.cost_c2, g.cost_c1, g.cost_c0)

    print("\nConverted PowerCase branches:")
    for br in case.branches[:10]:
        print(br.id, br.r, br.x, br.b_fr, br.b_to, br.rate_a, br.tap, br.shift)
        
        
audit_pypowsybl_units(network_raw, case)

network nominal_apparent_power: 100.0
PowerCase base_mva: 100.0

Raw PyPowSyBl generators:
     name energy_source  target_p   min_p   max_p          min_q  \
id                                                                 
B1-G              OTHER     232.4 -9999.0  9999.0 -1.797693e+308   
B2-G              OTHER      40.0 -9999.0  9999.0  -4.000000e+01   
B3-G              OTHER       0.0 -9999.0  9999.0   0.000000e+00   
B6-G              OTHER       0.0 -9999.0  9999.0  -6.000000e+00   
B8-G              OTHER       0.0 -9999.0  9999.0  -6.000000e+00   

              max_q  rated_s reactive_limits_kind  target_v  target_q  \
id                                                                      
B1-G  1.797693e+308      NaN              MIN_MAX   143.100     -16.9   
B2-G   5.000000e+01      NaN              MIN_MAX   141.075      42.4   
B3-G   4.000000e+01      NaN              MIN_MAX   136.350      23.4   
B6-G   2.400000e+01      NaN              MIN_MAX    12.840      12

In [5]:
bus_df = pd.DataFrame([bus.__dict__ for bus in case.buses])
branch_df = pd.DataFrame([branch.__dict__ for branch in case.branches])
gen_df = pd.DataFrame([gen.__dict__ for gen in case.generators])
load_df = pd.DataFrame([load.__dict__ for load in case.loads])
shunt_df = pd.DataFrame([shunt.__dict__ for shunt in case.shunts])

display(bus_df.head())
display(branch_df.head())
display(branch_df[branch_df["branch_type"] == "two_windings_transformer"])
display(gen_df.head())
display(load_df.head())
display(shunt_df.head())

,id,index,bus_type,vm0,va0,vmin,vmax,nominal_v_kv
0,VL10_0,1,1,1.051,-0.263545,0.9,1.1,12.0
1,VL11_0,2,1,1.057,-0.258134,0.9,1.1,12.0
2,VL12_0,3,1,1.055,-0.263021,0.9,1.1,12.0
3,VL13_0,4,1,1.050,-0.264592,0.9,1.1,12.0
4,VL14_0,5,1,1.036,-0.279951,0.9,1.1,12.0


,id,index,from_bus,to_bus,r,x,b,b_fr,b_to,g_fr,g_to,rate_a,tap,shift,angmin,angmax,status,branch_type
0,L1-2-1,1,VL1_0,VL2_0,0.01938,0.05917,0.0528,0.0264,0.0264,0.0,0.0,0.0,1.0,0.0,-1.570796,1.570796,1,line
1,L1-5-1,2,VL1_0,VL5_0,0.05403,0.22304,0.0492,0.0246,0.0246,0.0,0.0,0.0,1.0,0.0,-1.570796,1.570796,1,line
2,L10-11-1,3,VL10_0,VL11_0,0.08205,0.19207,0.0000,0.0000,0.0000,0.0,0.0,0.0,1.0,0.0,-1.570796,1.570796,1,line
3,L12-13-1,4,VL12_0,VL13_0,0.22092,0.19988,0.0000,0.0000,0.0000,0.0,0.0,0.0,1.0,0.0,-1.570796,1.570796,1,line
4,L13-14-1,5,VL13_0,VL14_0,0.17093,0.34802,0.0000,0.0000,0.0000,0.0,0.0,0.0,1.0,0.0,-1.570796,1.570796,1,line


,id,index,from_bus,to_bus,r,x,b,b_fr,b_to,g_fr,g_to,rate_a,tap,shift,angmin,angmax,status,branch_type
17,T4-7-1,18,VL4_0,VL7_0,0.0,0.20912,0.0,0.0,0.0,0.0,0.0,0.0,0.978,0.0,-1.570796,1.570796,1,two_windings_transformer
18,T4-9-1,19,VL4_0,VL9_0,0.0,0.55618,0.0,0.0,0.0,0.0,0.0,0.0,0.969,0.0,-1.570796,1.570796,1,two_windings_transformer
19,T5-6-1,20,VL5_0,VL6_0,0.0,0.25202,0.0,0.0,0.0,0.0,0.0,0.0,0.932,0.0,-1.570796,1.570796,1,two_windings_transformer


,id,index,bus,pg0,qg0,pmin,pmax,qmin,qmax,status,voltage_regulator_on
0,B1-G,1,VL1_0,2.324,-0.169,-99.99,99.99,-1.797693e+306,1.797693e+306,1,True
1,B2-G,2,VL2_0,0.400,0.424,-99.99,99.99,-4.000000e-01,5.000000e-01,1,True
2,B3-G,3,VL3_0,0.000,0.234,-99.99,99.99,0.000000e+00,4.000000e-01,1,True
3,B6-G,4,VL6_0,0.000,0.122,-99.99,99.99,-6.000000e-02,2.400000e-01,1,True
4,B8-G,5,VL8_0,0.000,0.174,-99.99,99.99,-6.000000e-02,2.400000e-01,1,True


,id,index,bus,pd,qd,status
0,B10-L,1,VL10_0,0.090,0.058,1
1,B11-L,2,VL11_0,0.035,0.018,1
2,B12-L,3,VL12_0,0.061,0.016,1
3,B13-L,4,VL13_0,0.135,0.058,1
4,B14-L,5,VL14_0,0.149,0.050,1


,id,index,bus,gs,bs,status
0,B9-SH,1,VL9_0,0.0,0.19,1


## M4-M6: Own Python DC PF And Newton AC PF

In [6]:
own_dc_pf = solve_dc_power_flow(case)
own_ac_pf = solve_ac_power_flow_newton(case)

display(solution_metrics(own_dc_pf))
display(solution_metrics(own_ac_pf))

,converged,mismatch_norm,iterations
0,True,1.262879e-15,None


,converged,mismatch_norm,iterations
0,True,4.974632e-13,4


In [7]:
own_bus_result_df = bus_df[["id", "nominal_v_kv", "bus_type"]].copy()
own_bus_result_df["vm_pu"] = own_ac_pf.vm
own_bus_result_df["vm_kv"] = own_ac_pf.vm * own_bus_result_df["nominal_v_kv"]
own_bus_result_df["va_rad"] = own_ac_pf.va
own_bus_result_df["va_deg"] = own_ac_pf.va * 180.0 / 3.141592653589793
display(own_bus_result_df.head())

own_branch_result_df = pd.DataFrame([
    {"branch_id": branch_id, **flow}
    for branch_id, flow in own_ac_pf.branch_flows.items()
])
display(own_branch_result_df.head())

,id,nominal_v_kv,bus_type,vm_pu,vm_kv,va_rad,va_deg
0,VL10_0,12.0,1,0.924814,11.097766,-0.261225,-14.967095
1,VL11_0,12.0,1,0.992252,11.907021,-0.270545,-15.501066
2,VL12_0,12.0,1,1.043623,12.523473,-0.291549,-16.704515
3,VL13_0,12.0,1,1.027852,12.334227,-0.288194,-16.512282
4,VL14_0,12.0,1,0.937961,11.255537,-0.289179,-16.568749


,branch_id,pf,qf,pt,qt
0,L1-2-1,1.604257,-0.212283,-1.559291,0.291078
1,L1-5-1,0.766675,0.125553,-0.737282,-0.056482
2,L10-11-1,-0.079577,-0.290511,0.088281,0.310885
3,L12-13-1,0.032876,0.046036,-0.032227,-0.045448
4,L13-14-1,0.107252,0.212811,-0.098064,-0.194103


## M7: Compare Own AC PF Against PyPowSyBl AC Load Flow

Controls are fixed to make the first comparison cleaner. Differences are expected while unit conversion and equipment-control assumptions are being refined.

In [8]:
network_lf = pn.create_ieee14()
buses_before = network_lf.get_buses().copy()
lines_before = network_lf.get_lines().copy()
generators_before = network_lf.get_generators().copy()

params = lf.Parameters(
    distributed_slack=True,
    use_reactive_limits=True,
    transformer_voltage_control_on=False,
    phase_shifter_regulation_on=False,
    shunt_compensator_voltage_control_on=False,
)

pypowsybl_lf_result = lf.run_ac(network_lf, parameters=params)
print(pypowsybl_lf_result)

buses_after = network_lf.get_buses().copy()
lines_after = network_lf.get_lines().copy()
generators_after = network_lf.get_generators().copy()

display(buses_before.join(buses_after, lsuffix="_before", rsuffix="_after").head())
display(lines_before.join(lines_after, lsuffix="_before", rsuffix="_after").head())
display(generators_before.join(generators_after, lsuffix="_before", rsuffix="_after").head())

[ComponentResult(connected_component_num=0, synchronous_component_num=0, status=CONVERGED, status_text=Converged, iteration_count=3, reference_bus_id='VL1_0', slack_bus_results=[SlackBusResult(id='VL1_0', active_power_mismatch=-0.006730108617913899)], distributed_active_power=0.0)]


,name_before,v_mag_before,v_angle_before,connected_component_before,synchronous_component_before,voltage_level_id_before,name_after,v_mag_after,v_angle_after,connected_component_after,synchronous_component_after,voltage_level_id_after
id,,,,,,,,,,,,
VL1_0,,143.100,0.00,0,0,VL1,,143.100000,0.000000,0,0,VL1
VL2_0,,141.075,-4.98,0,0,VL2,,141.075000,-4.982589,0,0,VL2
VL3_0,,136.350,-12.72,0,0,VL3,,136.350000,-12.725100,0,0,VL3
VL4_0,,137.565,-10.33,0,0,VL4,,137.385566,-10.312901,0,0,VL4
VL5_0,,137.700,-8.78,0,0,VL5,,137.634371,-8.773854,0,0,VL5


,name_before,r_before,x_before,g1_before,b1_before,g2_before,b2_before,p1_before,q1_before,i1_before,...,i1_after,p2_after,q2_after,i2_after,voltage_level1_id_after,voltage_level2_id_after,bus1_id_after,bus2_id_after,connected1_after,connected2_after
id,,,,,,,,,,,,,,,,,,,,,
L1-2-1,,3.532005,10.783732,0.0,0.000145,0.0,0.000145,NaN,NaN,NaN,...,638.289658,-152.585288,27.676249,634.645211,VL1,VL2,VL1_0,VL2_0,True,True
L1-5-1,,9.846967,40.649040,0.0,0.000135,0.0,0.000135,NaN,NaN,NaN,...,305.050421,-72.747509,2.229360,305.305359,VL1,VL5,VL1_0,VL5_0,True,True
L2-3-1,,8.563927,36.080033,0.0,0.000120,0.0,0.000120,NaN,NaN,NaN,...,300.079154,-70.914309,1.602233,300.350895,VL2,VL3,VL2_0,VL3_0,True,True
L2-4-1,,10.590547,32.134320,0.0,0.000093,0.0,0.000093,NaN,NaN,NaN,...,229.806083,-54.454838,3.020689,229.193286,VL2,VL4,VL2_0,VL4_0,True,True
L2-5-1,,10.379137,31.689630,0.0,0.000095,0.0,0.000095,NaN,NaN,NaN,...,169.972926,-40.612462,-2.099032,170.589018,VL2,VL5,VL2_0,VL5_0,True,True


,name_before,energy_source_before,target_p_before,min_p_before,max_p_before,min_q_before,max_q_before,rated_s_before,reactive_limits_kind_before,target_v_before,...,target_v_after,target_q_after,voltage_regulator_on_after,regulated_element_id_after,p_after,q_after,i_after,voltage_level_id_after,bus_id_after,connected_after
id,,,,,,,,,,,,,,,,,,,,,
B1-G,,OTHER,232.4,-9999.0,9999.0,-1.797693e+308,1.797693e+308,NaN,MIN_MAX,143.100,...,143.100,-16.9,True,B1-G,-232.4,16.549301,940.013774,VL1,VL1_0,True
B2-G,,OTHER,40.0,-9999.0,9999.0,-4.000000e+01,5.000000e+01,NaN,MIN_MAX,141.075,...,141.075,42.4,True,B2-G,-40.0,-43.557096,242.019762,VL2,VL2_0,True
B3-G,,OTHER,0.0,-9999.0,9999.0,0.000000e+00,4.000000e+01,NaN,MIN_MAX,136.350,...,136.350,23.4,True,B3-G,-0.0,-25.075347,106.177177,VL3,VL3_0,True
B6-G,,OTHER,0.0,-9999.0,9999.0,-6.000000e+00,2.400000e+01,NaN,MIN_MAX,12.840,...,12.840,12.2,True,B6-G,-0.0,-12.730942,572.446472,VL6,VL6_0,True
B8-G,,OTHER,0.0,-9999.0,9999.0,-6.000000e+00,2.400000e+01,NaN,MIN_MAX,21.800,...,21.800,17.4,True,B8-G,-0.0,-17.623450,466.738691,VL8,VL8_0,True


In [9]:
comparison = compare_with_pypowsybl(case, own_ac_pf, network_lf)

ac_pf_bus_compare_df = comparison["bus_comparison"]
ac_pf_branch_compare_df = comparison["branch_comparison"]
ac_pf_summary_df = comparison["summary"]

display(ac_pf_summary_df)
display(ac_pf_bus_compare_df.sort_values("vm_kv_diff", key=lambda s: s.abs(), ascending=False).head())
display(ac_pf_branch_compare_df.sort_values("pf_mw_diff", key=lambda s: s.abs(), ascending=False).head())

,max_vm_kv_diff,max_va_deg_diff,max_pf_mw_diff,max_qf_mvar_diff,max_pt_mw_diff,max_qt_mvar_diff
0,3.873429,1.642602,11.7581,123.769452,11.7581,249.877769


,bus_id,own_vm_pu,own_vm_kv,pypowsybl_vm_kv,vm_kv_diff,own_va_rad,own_va_deg,pypowsybl_va_deg,va_deg_diff
8,VL4_0,0.988979,133.512137,137.385566,-3.873429,-0.174978,-10.025504,-10.312901,0.287397
9,VL5_0,1.000534,135.072065,137.634371,-2.562307,-0.154038,-8.825727,-8.773854,-0.051873
13,VL9_0,0.903972,10.847666,12.671181,-1.823515,-0.251324,-14.399830,-14.938521,0.538691
0,VL10_0,0.924814,11.097766,12.611816,-1.514050,-0.261225,-14.967095,-15.097289,0.130193
11,VL7_0,0.958847,13.423858,14.861273,-1.437415,-0.225732,-12.933487,-13.359627,0.426140


,branch_id,branch_type,from_bus,to_bus,own_pf_mw,pypowsybl_p1_mw,pf_mw_diff,own_qf_mvar,pypowsybl_q1_mvar,qf_mvar_diff,own_pt_mw,pypowsybl_p2_mw,pt_mw_diff,own_qt_mvar,pypowsybl_q2_mvar,qt_mvar_diff
19,T5-6-1,two_windings_transformer,VL5_0,VL6_0,55.845422,44.087321,11.758100,4.939595,12.470681,-7.531086,-55.845422,-44.087321,-11.758100,1.933668,-8.049519,9.983188
9,L4-5-1,line,VL4_0,VL5_0,-52.377462,-61.158231,8.780769,-10.017448,15.823642,-25.841090,52.765611,61.672650,-8.907039,11.241789,-14.201004,25.442793
10,L6-11-1,line,VL6_0,VL11_0,13.518224,7.353277,6.164947,35.380728,3.560472,31.820256,-12.328142,-7.297904,-5.030239,-32.888549,-3.444514,-29.444035
4,L13-14-1,line,VL13_0,VL14_0,10.725212,5.643851,5.081361,21.281074,1.747173,19.533901,-9.806372,-5.589773,-4.216599,-19.410282,-1.637069,-17.773213
14,L7-9-1,line,VL7_0,VL9_0,23.522560,28.074178,-4.551617,-83.187090,5.778690,-88.965780,-23.522560,-28.074178,4.551617,53.809862,-4.976622,58.786484


## M8: PyPowSyBl Security Analysis To ML-Style Features And Labels

In [10]:
network_security = pn.create_ieee14()
security_result = run_n_minus_one_branch_study(
    network_security,
    include_transformers=True,
    max_loading_pct=100.0,
    parameters=params,
)

security_summary_df = security_result.contingency_summary
security_branches_df = security_result.contingency_branches
security_buses_df = security_result.contingency_buses

display(security_summary_df.head())
display(security_branches_df.head())
display(security_buses_df.head())

,contingency_id,scenario,status,classification,n_branch_violations,n_voltage_violations,max_branch_loading_pct,min_vm_kv,max_vm_kv
0,L1-2-1,outage_L1-2-1,ComponentStatus.CONVERGED,secure,0,0,NaN,11.345147,143.1
1,L1-5-1,outage_L1-5-1,ComponentStatus.CONVERGED,secure,0,0,NaN,12.389765,143.1
2,L2-3-1,outage_L2-3-1,ComponentStatus.CONVERGED,secure,0,0,NaN,12.353337,143.1
3,L2-4-1,outage_L2-4-1,ComponentStatus.CONVERGED,secure,0,0,NaN,12.383371,143.1
4,L2-5-1,outage_L2-5-1,ComponentStatus.CONVERGED,secure,0,0,NaN,12.410030,143.1


,scenario,branch_id,branch_type,bus1_id,bus2_id,voltage_level1_id,voltage_level2_id,p1_mw,q1_mvar,p2_mw,q2_mvar,s1_mva,s2_mva,rate_mva,loading1_pct,loading2_pct,loading_pct,connected1,connected2
0,outage_L1-2-1,L1-2-1,line,,,VL1,VL2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
1,outage_L1-2-1,L1-5-1,line,VL1_0,VL5_0,VL1,VL5,245.596252,66.550766,-214.281326,57.854141,254.453382,221.954023,NaN,NaN,NaN,NaN,True,True
2,outage_L1-2-1,L2-3-1,line,VL2_0,VL3_0,VL2,VL3,50.262334,1.602819,-48.950326,0.050699,50.287884,48.950352,NaN,NaN,NaN,NaN,True,True
3,outage_L1-2-1,L2-4-1,line,VL2_0,VL4_0,VL2,VL4,6.196483,11.658665,-6.060535,-14.248198,13.203063,15.483579,NaN,NaN,NaN,NaN,True,True
4,outage_L1-2-1,L2-5-1,line,VL2_0,VL5_0,VL2,VL5,-25.524081,24.036811,26.342769,-24.588362,35.060619,36.035108,NaN,NaN,NaN,NaN,True,True


,scenario,bus_id,voltage_level_id,connected_component,synchronous_component,vm_kv,va_deg
0,outage_L1-2-1,VL1_0,VL1,0,0,143.100000,0.000000
1,outage_L1-2-1,VL2_0,VL2,0,0,128.741496,-35.228188
2,outage_L1-2-1,VL3_0,VL3,0,0,125.159013,-41.580126
3,outage_L1-2-1,VL4_0,VL4,0,0,124.936671,-35.439335
4,outage_L1-2-1,VL7_0,VL7,0,0,13.643056,-38.680210


In [11]:
ml_feature_label_df = security_summary_df.copy()
ml_feature_label_df["label_secure"] = ml_feature_label_df["classification"].eq("secure").astype(int)
ml_feature_label_df["label_insecure"] = ml_feature_label_df["classification"].eq("insecure").astype(int)
ml_feature_label_df["label_non_converged"] = ~ml_feature_label_df["status"].astype(str).str.upper().str.contains("CONVERGED")

display(ml_feature_label_df.head())

,contingency_id,scenario,status,classification,n_branch_violations,n_voltage_violations,max_branch_loading_pct,min_vm_kv,max_vm_kv,label_secure,label_insecure,label_non_converged
0,L1-2-1,outage_L1-2-1,ComponentStatus.CONVERGED,secure,0,0,NaN,11.345147,143.1,1,0,False
1,L1-5-1,outage_L1-5-1,ComponentStatus.CONVERGED,secure,0,0,NaN,12.389765,143.1,1,0,False
2,L2-3-1,outage_L2-3-1,ComponentStatus.CONVERGED,secure,0,0,NaN,12.353337,143.1,1,0,False
3,L2-4-1,outage_L2-4-1,ComponentStatus.CONVERGED,secure,0,0,NaN,12.383371,143.1,1,0,False
4,L2-5-1,outage_L2-5-1,ComponentStatus.CONVERGED,secure,0,0,NaN,12.410030,143.1,1,0,False
